# Aperture Hardware Test

Exercise the motorized-aperture Tango interface on the real Spectra microscope. Discovery is read-only; every physical change is explicitly guarded.

## Safety

- Confirm that no one else is controlling the aperture mechanisms.
- Run one cell at a time and watch the microscope for errors.
- Keep `ALLOW_HARDWARE_CHANGES = False` during discovery.
- The default values make every physical-change cell a no-op, so **Run All** is read-only.

The notebook can restore the selected aperture and X/Y position captured for one mechanism. It does **not** automatically restore insertion or enabled state. If you enable, disable, insert, or retract a mechanism, record and restore that state manually.

### Run the servers

Make sure the microscope network/VPN and AutoScript server are available. From PowerShell in the repository root, synchronize the environment and open the server GUI:

```powershell
.\.venv\Scripts\python.exe startup_guis/server_gui.py
```

In the GUI:

1. Select `Spectra300.yaml`.
2. Confirm that **aperture** appears and is checked in the Devices list.
3. Click **Start**.
4. Wait for `asyncroscopy/aperture/default` to report `OK ready`.
5. Leave the GUI running while using this notebook.

The Spectra configuration registers the Tango device as:

```yaml
class_name: AutoScriptAPERTURE
module_name: asyncroscopy.instruments.electron_microscope.hardware.aperture_autoscript
```

`aperture.py` defines the vendor-neutral `APERTURE` Tango interface. `aperture_autoscript.py` provides the concrete `AutoScriptAPERTURE` server that implements that interface for the Spectra microscope and connects to AutoScript at `10.46.217.241:9095`.

Do not start `aperture.py` directly, manually register another aperture device, or launch a second aperture server while the GUI-managed server is running.

### Imports

In [ ]:
import os
import time

import numpy as np
import tango

# Global safety interlock used by every cell that can change hardware.
ALLOW_HARDWARE_CHANGES = True

### Ping aperture server

In [ ]:
DB_HOST = "10.46.217.241"
DB_PORT = 9094

os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"

aperture = tango.DeviceProxy("asyncroscopy/aperture/default")
aperture.set_timeout_millis(120_000)
aperture.ping()

print(aperture.name(), aperture.state())
print("Status:", aperture.status())
print("Mechanisms:", list(aperture.available_mechanisms))

### Read-only discovery

In [ ]:
def aperture_status():
    position = np.asarray(aperture.position, dtype=float)
    return {
        "mechanism": aperture.mechanism,
        "available_apertures": list(aperture.available_apertures),
        "selected_aperture": aperture.selected_aperture,
        "aperture_type": aperture.aperture_type,
        "aperture_diameter_m": float(aperture.aperture_diameter),
        "insertion_state": aperture.insertion_state,
        "enabled": bool(aperture.enabled),
        "retractable": bool(aperture.retractable),
        "position_m": position.tolist(),
    }

available_mechanisms = list(aperture.available_mechanisms)
if not available_mechanisms:
    raise RuntimeError("AutoScript reported no available aperture mechanisms")

print("Available mechanisms:", available_mechanisms)
print("Current mechanism:", aperture.mechanism)
aperture_status()

### Choose a mechanism and capture its starting state

Set `TARGET_MECHANISM` to an exact name printed above. This selects which mechanism later cells address; it does not move hardware.

In [ ]:
TARGET_MECHANISM = "C2"  # Example: "C2" or "Objective"

if TARGET_MECHANISM not in available_mechanisms:
    raise ValueError(f"Choose a mechanism from {available_mechanisms}")
aperture.mechanism = TARGET_MECHANISM

time.sleep(0.25)
starting_aperture = aperture.selected_aperture
starting_position = np.asarray(aperture.position, dtype=float).copy()

print("Starting aperture:", starting_aperture)
print("Starting position [m]:", starting_position)
aperture_status()

### Test aperture selection (physical change)

Choose an exact name from `available_apertures`. The default retains the current aperture.

In [ ]:
TARGET_APERTURE = starting_aperture

available_apertures = list(aperture.available_apertures)
print("Available apertures:", available_apertures)
if TARGET_APERTURE not in available_apertures:
    raise ValueError(f"Choose an aperture from {available_apertures}")

if TARGET_APERTURE == aperture.selected_aperture:
    print("No aperture-selection change requested.")
else:
    if not ALLOW_HARDWARE_CHANGES:
        raise RuntimeError("Set ALLOW_HARDWARE_CHANGES = True in the imports cell")
    aperture.selected_aperture = TARGET_APERTURE
    time.sleep(1.0)
    if aperture.selected_aperture != TARGET_APERTURE:
        raise RuntimeError("Aperture selection did not reach the requested value")

aperture_status()

### Test aperture position (physical change)

Offsets are in meters and limited here to 5 micrometers per axis. The default zero offset makes no move.

In [ ]:
POSITION_OFFSET = np.array([0.0e-6, 0.0e-6])  # [dx, dy] in meters
MAX_ABS_OFFSET = 50e-6

if POSITION_OFFSET.shape != (2,):
    raise ValueError("POSITION_OFFSET must be [dx, dy]")
if np.any(np.abs(POSITION_OFFSET) > MAX_ABS_OFFSET):
    raise ValueError(f"Each offset must be <= {MAX_ABS_OFFSET:g} m")

if not np.any(POSITION_OFFSET):
    print("No move requested; set a non-zero POSITION_OFFSET to test motion.")
else:
    if not ALLOW_HARDWARE_CHANGES:
        raise RuntimeError("Set ALLOW_HARDWARE_CHANGES = True in the imports cell")
    if starting_position.shape != (2,):
        raise RuntimeError("The selected mechanism did not report a two-axis starting position")
    target_position = starting_position + POSITION_OFFSET
    aperture.position = target_position.tolist()
    time.sleep(1.0)
    measured_position = np.asarray(aperture.position, dtype=float)
    print("Target [m]:", target_position)
    print("Measured [m]:", measured_position)

### Test commands (physical changes)

Enable only one flag at a time. Retraction is blocked unless the mechanism reports that it is retractable. `reset_positions` is omitted because it may change stored calibration.

In [ ]:
RUN_ENABLE = False
RUN_DISABLE = False
RUN_INSERT = False
RUN_RETRACT = False

requested = [RUN_ENABLE, RUN_DISABLE, RUN_INSERT, RUN_RETRACT]
if sum(requested) > 1:
    raise ValueError("Enable only one command flag at a time")
if any(requested) and not ALLOW_HARDWARE_CHANGES:
    raise RuntimeError("Set ALLOW_HARDWARE_CHANGES = True in the imports cell")
if RUN_RETRACT and not aperture.retractable:
    raise RuntimeError(f"{aperture.mechanism!r} is not retractable")

if RUN_ENABLE:
    aperture.enable()
elif RUN_DISABLE:
    aperture.disable()
elif RUN_INSERT:
    aperture.insert()
elif RUN_RETRACT:
    aperture.retract()
else:
    print("No command requested.")

if any(requested):
    time.sleep(1.0)
aperture_status()

### Restore the starting aperture and position

This restores only the selection and position captured for `TARGET_MECHANISM`. It does not call `reset_positions` or alter insertion/enabled state.

In [ ]:
RESTORE_STARTING_STATE = False

if not RESTORE_STARTING_STATE:
    print("Restore not requested.")
else:
    if not ALLOW_HARDWARE_CHANGES:
        raise RuntimeError("Set ALLOW_HARDWARE_CHANGES = True in the imports cell")
    aperture.mechanism = TARGET_MECHANISM
    if starting_aperture:
        aperture.selected_aperture = starting_aperture
    if starting_position.shape == (2,):
        aperture.position = starting_position.tolist()
    time.sleep(1.0)
    print("Starting selection and position restored.")

aperture_status()